<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/B_frozen_test_suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B — One frozen test suite for every model

**Goal.** Evaluate **every** saved model on **exactly the same** test subsets and produce **one** results table with one set of numbers (AUC with 95% bootstrap interval, accuracy, fakes caught, reals kept), so the thesis no longer contains several "clean" values for the same model.

**Test sets (built exactly like in your Phase 10 and Phase 7b notebooks):**
| name | content |
|---|---|
| `stylegan` | 5,000 real + 5,000 fake from the StyleGAN test split (seed 99) |
| `faceswap` | 5,000 real + 5,000 fake from the face-swap test split (seed 99) |
| `sd14_confounded` | old Stable Diffusion v1.4 held-out fakes + equal number of StyleGAN real images — **has the file-format shortcut; keep for reference only** |
| `DF_A_SD15`, `DF_A_SD21` | DiffusionFace: 3,000 real (DiffusionFace) + 3,000 fakes, seed 7 |
| `DF_B_SD15`, `DF_B_SD21` | the same fakes against 3,000 FFHQ real images |

**Models:** Phase 1, Phase 2, EfficientNet-B0, ensemble (P2 + EfficientNet), Phase 6, Phase 8 (frequency), Phase 9 (CLIP head), and the three seeds of Phase 2 and Phase 6 (reported as mean ± std).

**Needs on Drive** (`thesis_checkpoints/`): `best_model.pth`, `best_model_finetuned.pth`, `best_model_efficientnet.pth`, `best_model_mixed.pth`, `best_model_frequency.pth`, `best_model_clip_head.pth`, and `seed_runs/{p2,p6}_seed{0,1,2}.pth` + `seed_runs/diffusion_split.json`. Missing models are skipped with a message.

Runtime: GPU, roughly 30–60 minutes the first time (downloads + 12 model passes). Predictions are cached on Drive, so you can stop and resume.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
SEED_DIR  = f'{CHECKPOINT_DIR}/seed_runs'
OUT_DIR   = f'{CHECKPOINT_DIR}/frozen_suite'
CACHE_DIR = f'{OUT_DIR}/cache'
for d in [OUT_DIR, CACHE_DIR]: os.makedirs(d, exist_ok=True)
N_BOOT = 1000
print('Outputs go to:', OUT_DIR)

Tue Oct 20 23:16:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1 — Download the datasets (token typed at runtime, not stored)

In [ ]:
from getpass import getpass
os.environ['KAGGLE_API_TOKEN'] = getpass('Paste your Kaggle API token (not saved in the notebook): ')
!pip install -q kaggle transformers
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q -o 140k-real-and-fake-faces.zip -d data_raw
!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q -o deepfake-and-real-images.zip -d data_raw_2
!kaggle datasets download -d bwandowando/faces-dataset-using-stable-diffusion-v14
!unzip -q -o faces-dataset-using-stable-diffusion-v14.zip -d data_diffusion
print('Kaggle datasets ready.')

Paste your Kaggle API token (not saved in the notebook): ··········
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [02:04<00:00, 32.4MB/s]

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:58<00:00, 31.1MB/s]

Dataset URL: https://www.kaggle.com/datasets/bwandowando/faces-dataset-using-stable-diffusion-v14
License(s): CC0-1.0
100% 1.10G/1.10G [00:53<00:00, 22.0MB/s]

Kaggle datasets ready.


In [ ]:
# DiffusionFace (Chen et al., 2024) from Zenodo — same files and checksums as in your 07b notebook
import hashlib
WORK = '/content/diffface'; os.makedirs(WORK, exist_ok=True)
P1 = 'https://zenodo.org/records/10865300/files/{}?download=1'
P2 = 'https://zenodo.org/records/10865065/files/{}?download=1'
FILES = {
    'Real':     (P1.format('Real.tar'), 'Real.tar', '72379bcecf6382d7999cce59b33dfcdc'),
    'SD15_T2I': (P2.format('stable_diffusion_v_1_5_text2img_p3g7.tar'), 'stable_diffusion_v_1_5_text2img_p3g7.tar', '37a92f1973d6cc8941c8d98a195c09b4'),
    'SD21_T2I': (P2.format('stable_diffusion_v_2_1_text2img_p0g5.tar'), 'stable_diffusion_v_2_1_text2img_p0g5.tar', '55bd322166cfdb3bda9aa05cb5a545aa'),
}
def md5_of(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
for key, (url, fname, md5) in FILES.items():
    path = f'{WORK}/{fname}'
    if not os.path.exists(path):
        print('Downloading', fname); get_ipython().system(f'wget -c -q --show-progress -O "{path}" "{url}"')
    print(f'{fname:<48} md5', 'OK' if md5_of(path) == md5 else 'MISMATCH - delete the file and re-run this cell')
    dest = f'{WORK}/{key}'
    if not os.path.isdir(dest) or not os.listdir(dest):
        os.makedirs(dest, exist_ok=True); print('Extracting', fname); get_ipython().system(f'tar -xf "{path}" -C "{dest}"')
print('DiffusionFace ready.')

/content/diffface/R 100%[===================>]   3.23G  42.6MB/s    in 72s     
Real.tar                                         md5 OK
Extracting Real.tar
/content/diffface/s 100%[===================>]   2.78G  28.1MB/s    in 72s     
stable_diffusion_v_1_5_text2img_p3g7.tar         md5 OK
Extracting stable_diffusion_v_1_5_text2img_p3g7.tar
/content/diffface/s 100%[===================>]   2.90G  41.7MB/s    in 89s     
stable_diffusion_v_2_1_text2img_p0g5.tar         md5 OK
Extracting stable_diffusion_v_2_1_text2img_p0g5.tar
DiffusionFace ready.


### Step 2 — Build the frozen test sets (same functions and seeds as your notebooks)

In [ ]:
import random, json, numpy as np
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

def find_real_fake_folders(root):
    out = []
    for dp, dn, _ in os.walk(root):
        low = [d.lower() for d in dn]
        if 'real' in low and 'fake' in low: out.append(dp)
    return out
def find_split(root, kw):
    m = [x for x in find_real_fake_folders(root) if kw in x.lower()]
    if not m: raise FileNotFoundError(f"No '{kw}' folder under {root}")
    return m[0]
def find_class_dir(split_root, cls):
    for d in os.listdir(split_root):
        if d.lower() == cls: return os.path.join(split_root, d)
    raise FileNotFoundError(cls)
def find_all_images(root, exts=IMG_EXTS):
    out = []
    for dp, _, fn in os.walk(root):
        for f in fn:
            if f.lower().endswith(exts): out.append(os.path.join(dp, f))
    return sorted(out)

styleGAN_test = find_split('data_raw', 'test')
faceswap_test = find_split('data_raw_2', 'test')

def subset_items(source_root, n_per_class, seed=99):
    # identical logic to build_eval_subset in the Phase 10 notebook (one rng shared by both classes)
    rng = random.Random(seed); items = []
    for cls in ['real', 'fake']:
        cd = find_class_dir(source_root, cls); files = sorted(os.listdir(cd)); rng.shuffle(files)
        items += [(os.path.join(cd, f), int(cls == 'real')) for f in files[:n_per_class]]
    return items

EVAL = {}
EVAL['stylegan'] = subset_items(styleGAN_test, 5000)
EVAL['faceswap'] = subset_items(faceswap_test, 5000)

# old SD v1.4 set (confounded): same held-out split as the Phase 10 notebook
split_file = f'{SEED_DIR}/diffusion_split.json'
if os.path.exists(split_file):
    sp = json.load(open(split_file)); fakes = [os.path.join('data_diffusion', p) for p in sp['heldout']]
    rng = random.Random(7); real_dir = find_class_dir(styleGAN_test, 'real')
    reals = sorted(os.listdir(real_dir)); rng.shuffle(reals); reals = reals[:len(fakes)]
    EVAL['sd14_confounded'] = [(os.path.join(real_dir, f), 1) for f in reals] + [(p, 0) for p in sorted(fakes)]
else:
    print('diffusion_split.json not found -> sd14_confounded skipped')

# DiffusionFace (same as 07b: seed 7, 3,000 per class)
SEED, N = 7, 3000
def sample_paths(key, n):
    paths = find_all_images(f'{WORK}/{key}'); random.Random(SEED).shuffle(paths); return paths[:n]
real_df = sample_paths('Real', N)
fake_df = {k: sample_paths(k, N) for k in ['SD15_T2I', 'SD21_T2I']}
ffhq_dir = find_class_dir(styleGAN_test, 'real')
ffhq_all = sorted(os.path.join(ffhq_dir, f) for f in os.listdir(ffhq_dir)); random.Random(SEED).shuffle(ffhq_all); ffhq_real = ffhq_all[:N]
for pair, reals in [('A', real_df), ('B', ffhq_real)]:
    for fk, short in [('SD15_T2I', 'SD15'), ('SD21_T2I', 'SD21')]:
        n = min(len(reals), len(fake_df[fk]))
        EVAL[f'DF_{pair}_{short}'] = [(p, 1) for p in reals[:n]] + [(p, 0) for p in fake_df[fk][:n]]
for k, v in EVAL.items(): print(f'{k:<18} n={len(v):>6}  real={sum(y for _, y in v)}  fake={sum(1 - y for _, y in v)}')

stylegan           n= 10000  real=5000  fake=5000
faceswap           n= 10000  real=5000  fake=5000
sd14_confounded    n=  2030  real=1015  fake=1015
DF_A_SD15          n=  6000  real=3000  fake=3000
DF_A_SD21          n=  6000  real=3000  fake=3000
DF_B_SD15          n=  6000  real=3000  fake=3000
DF_B_SD21          n=  6000  real=3000  fake=3000


### Step 3 — Models and scoring (everything is cached per model × test set)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import CLIPModel
from PIL import Image
from tqdm.auto import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

std_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
class ToFreq:   # identical to the Phase 8 / Phase 10 notebooks
    def __call__(self, img):
        ch = [torch.log(torch.abs(torch.fft.fftshift(torch.fft.fft2(img[c]))) + 1e-8) for c in range(img.shape[0])]
        f = torch.stack(ch); return (f - f.mean()) / (f.std() + 1e-8)
freq_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), ToFreq()])
clip_tf = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC), transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])])

class ItemsDS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]; return self.tf(Image.open(p).convert('RGB')), int(y)

def load_resnet(path):
    ck = torch.load(path, map_location=device)
    m = models.resnet18(weights=None); m.fc = nn.Linear(m.fc.in_features, 2)
    m.load_state_dict(ck['model_state']); return m.to(device).eval(), ck['class_to_idx']
def load_effnet(path):
    ck = torch.load(path, map_location=device)
    m = models.efficientnet_b0(weights=None); m.classifier = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(m.classifier[1].in_features, 2))
    m.load_state_dict(ck['model_state']); return m.to(device).eval(), ck['class_to_idx']

_clip = {}
def load_clip_head(path):
    if 'm' not in _clip:
        _clip['m'] = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
    ck = torch.load(path, map_location=device)
    head = nn.Linear(_clip['m'].config.projection_dim, 2).to(device); head.load_state_dict(ck['head_state']); head.eval()
    def fwd(x):
        # Phase 9 used the un-normalised 512-d projected embedding
        return head(_clip['m'].visual_projection(_clip['m'].vision_model(pixel_values=x).pooler_output))
    return fwd, ck['class_to_idx']

def build(kind, path):
    if kind == 'resnet': m, c = load_resnet(path); return m, c, std_tf
    if kind == 'freq':   m, c = load_resnet(path); return m, c, freq_tf
    if kind == 'effnet': m, c = load_effnet(path); return m, c, std_tf
    if kind == 'clip':   f, c = load_clip_head(path); return f, c, clip_tf
    raise ValueError(kind)

@torch.no_grad()
def p_real_for(fwd, real_idx, items, tf):
    loader = DataLoader(ItemsDS(items, tf), batch_size=64, shuffle=False, num_workers=4)
    p, y = [], []
    for x, t in loader:
        p.append(torch.softmax(fwd(x.to(device)), 1)[:, real_idx].cpu().numpy()); y.append(t.numpy())
    return np.concatenate(p), np.concatenate(y)

MODELS = {   # key: (display name, kind, checkpoint)
    'p1':     ('Phase 1  frozen ResNet18',        'resnet', f'{CHECKPOINT_DIR}/best_model.pth'),
    'p2':     ('Phase 2  ResNet18 layer4',        'resnet', f'{CHECKPOINT_DIR}/best_model_finetuned.pth'),
    'eff':    ('Phase 4  EfficientNet-B0',        'effnet', f'{CHECKPOINT_DIR}/best_model_efficientnet.pth'),
    'p6':     ('Phase 6  StyleGAN+face-swap',     'resnet', f'{CHECKPOINT_DIR}/best_model_mixed.pth'),
    'p8':     ('Phase 8  frequency (FFT)',        'freq',   f'{CHECKPOINT_DIR}/best_model_frequency.pth'),
    'p9clip': ('Phase 9  CLIP ViT-B/32 + head',   'clip',   f'{CHECKPOINT_DIR}/best_model_clip_head.pth'),
}
for s in [0, 1, 2]:
    MODELS[f'p2_seed{s}'] = (f'Phase 2 seed {s}', 'resnet', f'{SEED_DIR}/p2_seed{s}.pth')
    MODELS[f'p6_seed{s}'] = (f'Phase 6 seed {s}', 'resnet', f'{SEED_DIR}/p6_seed{s}.pth')

SCORES = {}   # (model_key, set_name) -> (p_real, y_real)
for key, (disp, kind, path) in MODELS.items():
    todo = [s for s in EVAL if not os.path.exists(f'{CACHE_DIR}/{key}__{s}.npz')]
    if todo:
        if not os.path.exists(path): print('SKIP (checkpoint missing):', disp, path); continue
        fwd, cidx, tf = build(kind, path)
        for s in todo:
            p, y = p_real_for(fwd, cidx['real'], EVAL[s], tf)
            np.savez(f'{CACHE_DIR}/{key}__{s}.npz', p=p, y=y)
        del fwd; torch.cuda.empty_cache()
    for s in EVAL:
        f = f'{CACHE_DIR}/{key}__{s}.npz'
        if os.path.exists(f): z = np.load(f); SCORES[(key, s)] = (z['p'], z['y'])
    print('done', disp)

# ensemble = average of the Phase 2 and EfficientNet probabilities (as in Phase 4)
for s in EVAL:
    if ('p2', s) in SCORES and ('eff', s) in SCORES:
        SCORES[('ens', s)] = ((SCORES[('p2', s)][0] + SCORES[('eff', s)][0]) / 2, SCORES[('p2', s)][1])
MODELS['ens'] = ('Phase 4  ensemble (P2 + EffNet)', None, None)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


done Phase 1  frozen ResNet18
done Phase 2  ResNet18 layer4
done Phase 4  EfficientNet-B0
done Phase 6  StyleGAN+face-swap
done Phase 8  frequency (FFT)


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

done Phase 9  CLIP ViT-B/32 + head
done Phase 2 seed 0
done Phase 6 seed 0
done Phase 2 seed 1
done Phase 6 seed 1
done Phase 2 seed 2
done Phase 6 seed 2


### Step 4 — Metrics, bootstrap intervals, and a reproduction check

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

def metrics(y, p, n_boot=N_BOOT, seed=0):
    pred = p >= 0.5
    real_recall = float(pred[y == 1].mean()); fake_recall = float((~pred[y == 0]).mean())
    r = np.random.default_rng(seed); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; a = []
    for _ in range(n_boot):
        ii = np.concatenate([r.choice(pos, len(pos)), r.choice(neg, len(neg))]); a.append(roc_auc_score(y[ii], p[ii]))
    lo, hi = np.percentile(a, [2.5, 97.5])
    return dict(n=len(y), acc=float((pred == y).mean()), auc=float(roc_auc_score(y, p)), auc_lo=float(lo), auc_hi=float(hi),
                fake_recall=fake_recall, real_recall=real_recall, balanced_acc=(fake_recall + real_recall) / 2)

rows = []
for (key, s), (p, y) in SCORES.items():
    rows.append(dict(model_key=key, model=MODELS[key][0], test_set=s, **metrics(y, p)))
long = pd.DataFrame(rows); long.to_csv(f'{OUT_DIR}/results_long.csv', index=False)

# --- reproduction check against numbers you already have (tolerance 0.004 AUC)
KNOWN = [('p2', 'DF_A_SD15', 0.530), ('p2', 'DF_B_SD21', 0.498), ('p6', 'DF_A_SD15', 0.459), ('p6', 'DF_B_SD15', 0.498)]
for key, s, ref in KNOWN:
    r = long[(long.model_key == key) & (long.test_set == s)]
    if len(r): v = float(r.auc.iloc[0]); print(f'{key:>3} {s:<10} AUC {v:.3f}  (07b notebook: {ref:.3f})  ', 'OK' if abs(v - ref) <= 0.004 else '<-- DIFFERENT, tell me')

 p2 DF_A_SD15  AUC 0.530  (07b notebook: 0.530)   OK
 p2 DF_B_SD21  AUC 0.498  (07b notebook: 0.498)   OK
 p6 DF_A_SD15  AUC 0.459  (07b notebook: 0.459)   OK
 p6 DF_B_SD15  AUC 0.498  (07b notebook: 0.498)   OK


In [ ]:
SETS_MAIN = [s for s in ['stylegan', 'faceswap', 'DF_A_SD15', 'DF_A_SD21', 'DF_B_SD15', 'DF_B_SD21', 'sd14_confounded'] if s in EVAL]
single = ['p1', 'p2', 'eff', 'ens', 'p6', 'p8', 'p9clip']
def cell(r): return f"{r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}]"
tab = {}
for key in single:
    d = {}
    for s in SETS_MAIN:
        r = long[(long.model_key == key) & (long.test_set == s)]
        d[s] = cell(r.iloc[0]) if len(r) else '-'
    tab[MODELS[key][0]] = d
auc_table = pd.DataFrame(tab).T; auc_table.to_csv(f'{OUT_DIR}/auc_table_single_runs.csv');
print('AUC with 95% bootstrap interval (single runs, original checkpoints)'); auc_table

AUC with 95% bootstrap interval (single runs, original checkpoints)


,stylegan,faceswap,DF_A_SD15,DF_A_SD21,DF_B_SD15,DF_B_SD21,sd14_confounded
Phase 1 frozen ResNet18,"0.890 [0.883, 0.895]","0.490 [0.479, 0.501]","0.535 [0.519, 0.549]","0.496 [0.481, 0.510]","0.496 [0.480, 0.510]","0.459 [0.445, 0.474]","0.368 [0.344, 0.393]"
Phase 2 ResNet18 layer4,"0.999 [0.999, 0.999]","0.525 [0.514, 0.537]","0.530 [0.514, 0.545]","0.506 [0.490, 0.520]","0.523 [0.507, 0.537]","0.498 [0.483, 0.512]","0.282 [0.263, 0.303]"
Phase 4 EfficientNet-B0,"0.986 [0.985, 0.988]","0.500 [0.488, 0.512]","0.496 [0.480, 0.509]","0.492 [0.477, 0.506]","0.536 [0.521, 0.549]","0.532 [0.518, 0.546]","0.391 [0.366, 0.415]"
Phase 4 ensemble (P2 + EffNet),"0.999 [0.999, 0.999]","0.499 [0.487, 0.510]","0.498 [0.483, 0.512]","0.492 [0.477, 0.506]","0.537 [0.523, 0.551]","0.531 [0.516, 0.545]","0.385 [0.360, 0.408]"
Phase 6 StyleGAN+face-swap,"0.995 [0.995, 0.996]","0.927 [0.922, 0.932]","0.459 [0.444, 0.474]","0.466 [0.452, 0.482]","0.498 [0.482, 0.512]","0.504 [0.489, 0.518]","0.563 [0.539, 0.588]"
Phase 8 frequency (FFT),"0.917 [0.912, 0.922]","0.559 [0.547, 0.570]","0.319 [0.305, 0.332]","0.290 [0.277, 0.303]","0.121 [0.112, 0.129]","0.122 [0.113, 0.131]","0.302 [0.280, 0.326]"
Phase 9 CLIP ViT-B/32 + head,"0.987 [0.985, 0.988]","0.486 [0.475, 0.497]","0.303 [0.289, 0.316]","0.387 [0.374, 0.400]","0.463 [0.449, 0.479]","0.547 [0.531, 0.563]","0.232 [0.214, 0.251]"


In [ ]:
# Seed means (std with ddof=1, like in the thesis)
def seed_block(prefix):
    out = {}
    for s in SETS_MAIN:
        sub = long[(long.model_key.isin([f'{prefix}_seed{i}' for i in range(3)])) & (long.test_set == s)]
        if len(sub) == 3:
            out[s] = f"AUC {sub.auc.mean():.3f} ± {sub.auc.std(ddof=1):.3f} | acc {sub.acc.mean():.4f} ± {sub.acc.std(ddof=1):.4f}"
    return out
seed_table = pd.DataFrame({'Phase 2 (3 seeds)': seed_block('p2'), 'Phase 6 (3 seeds)': seed_block('p6')}).T
seed_table.to_csv(f'{OUT_DIR}/seed_means.csv'); seed_table

,stylegan,faceswap,DF_A_SD15,DF_A_SD21,DF_B_SD15,DF_B_SD21,sd14_confounded
Phase 2 (3 seeds),AUC 0.999 ± 0.000 | acc 0.9857 ± 0.0022,AUC 0.502 ± 0.014 | acc 0.4924 ± 0.0034,AUC 0.528 ± 0.030 | acc 0.4979 ± 0.0012,AUC 0.530 ± 0.042 | acc 0.4999 ± 0.0023,AUC 0.516 ± 0.075 | acc 0.4986 ± 0.0037,AUC 0.519 ± 0.017 | acc 0.5006 ± 0.0013,AUC 0.305 ± 0.018 | acc 0.4956 ± 0.0015
Phase 6 (3 seeds),AUC 0.996 ± 0.000 | acc 0.9694 ± 0.0031,AUC 0.898 ± 0.011 | acc 0.7908 ± 0.0067,AUC 0.496 ± 0.025 | acc 0.4959 ± 0.0036,AUC 0.493 ± 0.033 | acc 0.4991 ± 0.0072,AUC 0.515 ± 0.021 | acc 0.5005 ± 0.0005,AUC 0.512 ± 0.016 | acc 0.5037 ± 0.0045,AUC 0.629 ± 0.055 | acc 0.5420 ± 0.0246


In [ ]:
# Accuracy / fakes caught / reals kept for every model x set (for the appendix)
detail = long.assign(AUC=long.auc.round(3), acc=long.acc.round(3), fakes_caught=long.fake_recall.round(3), reals_kept=long.real_recall.round(3),
                     balanced_acc=long.balanced_acc.round(3))[['model', 'test_set', 'n', 'AUC', 'acc', 'fakes_caught', 'reals_kept', 'balanced_acc']]
detail.to_csv(f'{OUT_DIR}/results_detail.csv', index=False)
print('Files in', OUT_DIR); print(os.listdir(OUT_DIR))
detail.head(20)

Files in /content/drive/MyDrive/thesis_checkpoints/frozen_suite
['cache', 'results_long.csv', 'auc_table_single_runs.csv', 'seed_means.csv', 'results_detail.csv']


,model,test_set,n,AUC,acc,fakes_caught,reals_kept,balanced_acc
0,Phase 1 frozen ResNet18,stylegan,10000,0.890,0.810,0.806,0.814,0.810
1,Phase 1 frozen ResNet18,faceswap,10000,0.490,0.490,0.054,0.926,0.490
2,Phase 1 frozen ResNet18,sd14_confounded,2030,0.368,0.446,0.083,0.809,0.446
3,Phase 1 frozen ResNet18,DF_A_SD15,6000,0.535,0.502,0.149,0.855,0.502
4,Phase 1 frozen ResNet18,DF_A_SD21,6000,0.496,0.492,0.129,0.855,0.492
5,Phase 1 frozen ResNet18,DF_B_SD15,6000,0.496,0.481,0.149,0.813,0.481
6,Phase 1 frozen ResNet18,DF_B_SD21,6000,0.459,0.471,0.129,0.813,0.471
7,Phase 2 ResNet18 layer4,stylegan,10000,0.999,0.988,0.991,0.984,0.988
8,Phase 2 ResNet18 layer4,faceswap,10000,0.525,0.498,0.018,0.978,0.498
9,Phase 2 ResNet18 layer4,sd14_confounded,2030,0.282,0.496,0.002,0.990,0.496


### What to send back
Paste into the chat: (1) the **reproduction check** lines, (2) the **single-run AUC table**, (3) the **seed means table**. I will then replace the thesis' Table 4.18 and the diffusion numbers so every row comes from the same test subsets, and make DiffusionFace the main diffusion evidence.

If a reproduction line says *DIFFERENT*, stop and send it first — it means a subset or preprocessing step differs from your earlier notebooks.